# CIFAR-10 extraction: released target, then training from scratch

This notebook runs two separate experiments in order:

1. Download or reuse OpenAI's released CIFAR-10 Improved DDPM checkpoint, show unguided samples, and audit it.
2. Initialize a separate target from scratch, train or resume it, and audit that target.

The released target was trained on all 50,000 CIFAR-10 training images. Here its audit uses only the selected reference prefix, not its entire training set. The fresh target trains only on the selected prefix. Neither experiment reproduces Carlini's published extraction rates.

Install `requirements.txt` in the notebook kernel environment. `train_config.yaml` selects the target profile; `audit.yaml` controls both attacks. The default `smoke` target uses a small training budget; audit sample counts are configured independently.


In [ ]:
import gc
import json
import os
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

# Required before CUDA initializes; also makes repeated seeded samples comparable.
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import matplotlib.pyplot as plt
import torch
import torchvision
import yaml

candidates = [Path.cwd(), *Path.cwd().parents]
example_dir = next(
    (path / 'examples' / 'extraction' / 'cifar10' for path in candidates if (path / 'examples' / 'extraction' / 'cifar10' / 'audit.yaml').exists()),

    Path.cwd(),
)
if not (example_dir / 'train_config.yaml').exists():
    raise FileNotFoundError('Run this notebook from the LeakPro checkout or examples/extraction/cifar10 directory.')
repo_root = example_dir.parents[2]
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(example_dir))
os.chdir(example_dir)

from leakpro import LeakPro
from cifar10_handler import CIFAR10ExtractionHandler, load_audit_config
from cifar10_model import (
    RunProfile,
    ensure_released_checkpoint,
    load_pretrained_target,
    load_cifar10,
    make_adapter,
    make_feature_extractor,
    seed_everything,
    select_device,
    sha256_file,
    sha256_mapping,
    sha256_module_state,
    sha256_tensor,
    train_or_load_target,
)

train_config = yaml.safe_load(Path('train_config.yaml').read_text(encoding='utf-8'))
profile_name = os.getenv('LEAKPRO_CIFAR_PROFILE', train_config['run']['profile'])
if profile_name not in train_config['profiles']:
    raise ValueError(f'Unknown profile {profile_name!r}; choose from {sorted(train_config["profiles"])}.')
profile_values = dict(train_config['profiles'][profile_name])
profile = RunProfile(name=profile_name, seed=int(train_config['run']['random_seed']), **profile_values)
audit_config_value = os.getenv(
    'LEAKPRO_CIFAR_AUDIT_CONFIG',
    train_config['run']['audit_config'],
)
audit_config_path = Path(audit_config_value)
device = select_device(os.getenv('LEAKPRO_CIFAR_DEVICE', train_config['run']['device']))
seed_everything(profile.seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
data_dir = Path(os.getenv('LEAKPRO_CIFAR_DATA_DIR', train_config['run']['data_dir']))
target_root = Path(os.getenv('LEAKPRO_CIFAR_TARGET_DIR', train_config['run']['target_dir']))
target_dir = target_root / profile.name
target_dir.mkdir(parents=True, exist_ok=True)
notebook_source_hash = sha256_mapping({'cells': [
    {'cell_type': cell['cell_type'], 'source': cell['source']}
    for cell in json.loads(Path('main.ipynb').read_text(encoding='utf-8'))['cells']
]})
print({
    'profile': profile.name,
    'device': str(device),
    'data_dir': str(data_dir),
    'target_dir': str(target_dir),
    'audit_config': str(audit_config_path),
})

## Prepare the reference images

Both audits use the same deterministic CIFAR-10 training prefix. Class labels are not passed to either unconditional target. SIDE, when enabled in the audit YAML, derives its own labels from generated images.

In [ ]:
train_dataset, reference_images = load_cifar10(profile, target_dir, data_dir)
assert reference_images.shape == (profile.reference_size, 3, 32, 32)
assert reference_images.dtype == torch.float32
assert torch.isfinite(reference_images).all()
assert float(reference_images.min()) >= -1.0 and float(reference_images.max()) <= 1.0

figure, axes = plt.subplots(1, 8, figsize=(12, 2))
for axis, image in zip(axes, reference_images[:8].add(1.0).div(2.0)):
    axis.imshow(image.permute(1, 2, 0))
    axis.axis('off')
figure.suptitle(f'Reference prefix: {profile.reference_size} CIFAR-10 training images')
plt.show()

feature_extractor, feature_transform = make_feature_extractor()

## Shared audit and reporting

Each stage gets its own handler, target identity, resolved audit YAML, results directory, and manifest. Attack parameters come from the selected audit YAML. Only the runtime target identity and stage output directory are changed. A Carlini-only attack list is supported.

A candidate's small pixel distance does not by itself establish that the image is a recognizable copy. Inspect the unguided samples and candidate pairs.

In [ ]:
def show_nearest_matches(result, title, maximum=6):
    records = [record for record in result.candidates if record.nearest_reference_index is not None]
    records.sort(key=lambda record: record.nearest_reference_distance)
    records = records[:maximum]
    if not records:
        print(f'{title}: no qualifying candidates')
        return
    figure, axes = plt.subplots(len(records), 2, figsize=(4, 2 * len(records)), squeeze=False)
    for row, record in enumerate(records):
        generated = result.images[record.image_index].detach().cpu()
        reference = reference_images[record.nearest_reference_index].add(1.0).div(2.0)
        axes[row, 0].imshow(generated.permute(1, 2, 0).clamp(0.0, 1.0))
        axes[row, 0].set_title(f'generated, d={record.nearest_reference_distance:.4f}')
        axes[row, 1].imshow(reference.permute(1, 2, 0).clamp(0.0, 1.0))
        axes[row, 1].set_title(f'reference #{record.nearest_reference_index}')
        axes[row, 0].axis('off')
        axes[row, 1].axis('off')
    figure.suptitle(title)
    figure.tight_layout()
    plt.show()


def audit_target(stage, model, diffusion, checkpoint_path, stage_profile, training_origin, training_seconds=0.0):
    class StageHandler(CIFAR10ExtractionHandler):
        """Keep each audit's configured objects separate."""

    stage_dir = target_dir / stage
    stage_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_hash = sha256_file(checkpoint_path)
    adapter = make_adapter(model, diffusion, device)
    samples = adapter.sample(4, conditions=None, seed=profile.seed + 10)
    repeated = adapter.sample(4, conditions=None, seed=profile.seed + 10)
    assert samples.shape == (4, 3, 32, 32)
    assert torch.isfinite(samples).all()
    torch.testing.assert_close(samples, repeated)
    figure, axes = plt.subplots(1, 4, figsize=(7, 2))
    for axis, image in zip(axes, samples.detach().cpu().add(1.0).div(2.0)):
        axis.imshow(image.permute(1, 2, 0).clamp(0.0, 1.0))
        axis.axis('off')
    figure.suptitle(f'{stage}: unguided DDIM samples')
    plt.show()

    identity_components = {
        'experiment_stage': stage,
        'training_origin': training_origin,
        'target_checkpoint_sha256': checkpoint_hash,
        'model_source_sha256': sha256_file(Path('cifar10_model.py')),
        'handler_source_sha256': sha256_file(Path('cifar10_handler.py')),
        'notebook_source_sha256': notebook_source_hash,
        'sampling_steps': stage_profile.sampling_steps,
        'side_feature_state_sha256': sha256_module_state(feature_extractor),
        'side_feature_transform': 'resize-224-bilinear-align-corners-false-imagenet-normalization-v1',
        'authorized_references_sha256': sha256_tensor(reference_images),
    }
    target_fingerprint = f"sha256:{sha256_mapping(identity_components)}"
    StageHandler.configure(adapter=adapter, references=reference_images,
                           feature_extractor=feature_extractor, feature_transform=feature_transform)
    audit_config = load_audit_config(
        audit_config_path,
        target_fingerprint=target_fingerprint,
    )
    audit_output = Path(audit_config['audit']['output_dir']) / stage
    audit_config['audit']['output_dir'] = str(audit_output)
    runtime_config_path = stage_dir / 'audit.yaml'
    runtime_config_path.write_text(yaml.safe_dump(audit_config, sort_keys=False), encoding='utf-8')
    attack_configs = {entry['attack']: entry for entry in audit_config['audit']['attack_list']}
    audit_started = time.perf_counter()
    results = LeakPro(StageHandler, str(runtime_config_path)).run_audit()
    audit_seconds = time.perf_counter() - audit_started
    assert len(results) == len(attack_configs)
    for result in results:
        if result.metrics.get('mode') == 'unconditional_reference_audit':
            assert result.metrics['images_generated'] == attack_configs['carlini_diffusion']['num_unconditional_generations']
        else:
            assert result.metrics['images_generated'] == attack_configs['side']['num_generations']
            assert result.execution_trace[-1]['guidance_calls'] > 0
        result_dir = audit_output / 'results' / result.id
        assert (result_dir / 'result.json').exists()
        assert (result_dir / 'candidates.npz').exists()
        print(stage, result.name, result.metrics)
        show_nearest_matches(result, f'{stage}: {result.name}')
    manifest = {
        'stage': stage,
        'model_configuration': {key: getattr(stage_profile, key) for key in (
            'model_channels', 'num_res_blocks', 'dropout', 'timesteps', 'sampling_steps'
        )},
        'scratch_training_configuration': asdict(profile) if stage == 'from_scratch' else None,
        'training_origin': training_origin,
        'reference_scope': {'dataset': 'CIFAR-10 train', 'selection': 'prefix', 'count': profile.reference_size},
        'checkpoint': str(checkpoint_path),
        'checkpoint_sha256': checkpoint_hash,
        'target_fingerprint': target_fingerprint,
        'identity_components': identity_components,
        'device': str(device),
        'torch': torch.__version__,
        'torchvision': torchvision.__version__,
        'training_seconds': training_seconds,
        'audit_seconds': audit_seconds,
        'source_audit_config': str(audit_config_path),
        'runtime_audit_config': str(runtime_config_path),
        'audit_config': audit_config,
        'result_ids': [result.id for result in results],
    }
    manifest_path = stage_dir / 'run_manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')
    print({'stage': stage, 'manifest': str(manifest_path), 'audit_seconds': round(audit_seconds, 2)})
    return results

## 1. Load and audit the released checkpoint

This is the [official Improved DDPM CIFAR-10 hybrid-loss checkpoint](https://github.com/openai/improved-diffusion#models-and-hyperparameters), trained for 500,000 updates. Its download and cached file are checked against a fixed SHA-256 digest before loading. Its architecture and noise schedule remain the released configuration even when the scratch profile is `smoke`.

The profile fields describing scratch epochs and learning rate do not describe how this released checkpoint was trained. The manifest records its known training origin separately.

In [ ]:
released_profile = replace(profile, name='released', model_channels=128, num_res_blocks=3,
                           dropout=0.3, timesteps=4000,
                           sampling_steps=int(train_config['released']['sampling_steps']))
released_checkpoint_path = ensure_released_checkpoint(Path(os.getenv(
    'LEAKPRO_CIFAR_RELEASED_CHECKPOINT', train_config['released']['checkpoint_path']
)))
released_model, released_diffusion = load_pretrained_target(released_profile, released_checkpoint_path, device)
released_results = audit_target(
    'released', released_model, released_diffusion, released_checkpoint_path, released_profile,
    {'kind': 'official_released_checkpoint', 'dataset': 'CIFAR-10 train', 'training_images': 50000,
     'optimizer_updates': 500000, 'reference_set_is_partial': profile.reference_size < 50000},
)
# Results retain CPU images; release the model before scratch training.
del released_model, released_diffusion
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

## 2. Train from scratch and audit

This stage never initializes from the released weights. It creates a fresh official U-Net, or resumes/reloads a compatible checkpoint from this scratch stage's directory. The demonstration uses effective batch size 128, microbatch 32, learning rate `0.0001`, and EMA `0.999` for the shorter budget. These budget and EMA choices are experimental, not the full published training recipe.

A separate resume checkpoint is saved every 100 completed epochs. `force_retrain: true` discards only this stage's completed/resume checkpoints. The selected configuration must match an existing scratch checkpoint to resume it. To preserve an older experiment when changing training settings, choose a new `target_dir`.

Inspect unguided samples before interpreting extraction results.

In [ ]:
scratch_dir = target_dir / 'from_scratch'
force_retrain_env = os.getenv('LEAKPRO_CIFAR_FORCE_RETRAIN')
force_retrain = force_retrain_env == '1' if force_retrain_env is not None else bool(train_config['run']['force_retrain'])
training_started = time.perf_counter()
scratch_model, scratch_diffusion, scratch_checkpoint_path, epoch_losses = train_or_load_target(
    profile, train_dataset, scratch_dir, device, force_retrain=force_retrain
)
training_seconds = time.perf_counter() - training_started
if epoch_losses:
    plt.figure(figsize=(6, 3))
    plt.plot(epoch_losses)
    plt.xlabel('Epoch')
    plt.ylabel('Improved DDPM hybrid loss')
    plt.yscale('log')
    plt.title('From-scratch target training loss')
    plt.show()
scratch_results = audit_target(
    'from_scratch', scratch_model, scratch_diffusion, scratch_checkpoint_path, profile,
    {'kind': 'from_scratch', 'dataset': 'CIFAR-10 train', 'selection': 'prefix',
     'training_images': profile.train_size, 'epochs': profile.epochs,
     'optimizer_updates': profile.epochs * ((profile.train_size + profile.train_batch_size - 1) // profile.train_batch_size)},
    training_seconds=training_seconds,
)